# IMDb Sentiment Analysis

A machine learning project for classifying IMDb movie reviews as **positive** or **negative**.

This notebook compares **Logistic Regression** and **Linear SVM** using **TF-IDF** text features.

## 1. Import Libraries

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

## 2. Load and Inspect the Dataset

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.describe(include="all")

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [4]:
df.isna().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


## 3. Prepare the Data

In [6]:
X = df["review"]
y = df["sentiment"].map({"positive": 1, "negative": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Training samples: 40000
Testing samples: 10000


## 4. Build the Machine Learning Pipelines

Both models use TF-IDF with:
- English stop-word removal
- up to 20,000 features
- unigrams and bigrams

The SVM pipeline uses `min_df=2`, while the Logistic Regression pipeline uses `min_df=5`.

In [7]:
pipe_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            max_features=20000,
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "model",
        LinearSVC(
            C=0.3,
            loss="squared_hinge",
            class_weight=None,
            dual=False,
            max_iter=5000
        )
    )
])

In [8]:
pipe_log = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            stop_words="english",
            max_features=20000,
            ngram_range=(1, 2),
            min_df=5
        )
    ),
    (
        "model",
        LogisticRegression(max_iter=200)
    )
])

## 5. Train the Models

In [9]:
pipe_log.fit(X_train, y_train)
pipe_svm.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english')),
                ('model', LinearSVC(C=0.3, dual=False, max_iter=5000))])

## 6. Generate Predictions

In [10]:
y_pred_log = pipe_log.predict(X_test)
y_pred_svm = pipe_svm.predict(X_test)

## 7. Evaluate Logistic Regression

In [11]:
log_accuracy = accuracy_score(y_test, y_pred_log)

print(f"Accuracy: {log_accuracy:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_log))

Accuracy: 0.9001

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.89      0.90      5000
           1       0.89      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



## 8. Evaluate Linear SVM

In [12]:
svm_accuracy = accuracy_score(y_test, y_pred_svm)

print(f"Accuracy: {svm_accuracy:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

Accuracy: 0.9053

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.90      0.90      5000
           1       0.90      0.91      0.91      5000

    accuracy                           0.91     10000
   macro avg       0.91      0.91      0.91     10000
weighted avg       0.91      0.91      0.91     10000



## 9. Confusion Matrix — Linear SVM

In [13]:
confusion = confusion_matrix(y_test, y_pred_svm)
print(confusion)

[[4491  509]
 [ 438 4562]]


## 10. Custom Review Predictions

In [14]:
sample_review = "I hate that"

prediction = pipe_log.predict([sample_review])[0]
print("Positive" if prediction == 1 else "Negative")

Positive


In [15]:
sample_review = "That was terrible"

prediction = pipe_svm.predict([sample_review])[0]
print("Positive" if prediction == 1 else "Negative")

Negative


## 11. Conclusion

In the original experiment, **Linear SVM** achieved a slightly higher accuracy than **Logistic Regression**:

| Model | Accuracy |
|---|---:|
| Logistic Regression | 90.01% |
| Linear SVM | 90.52% |

Therefore, Linear SVM performed best on this test set.